# 04 Molecular Property Prediction: From Baselines to Random Forests

This section uses ESOL as a regression task: input molecular features, predict logS.

Application scenario: in synthesis, drug discovery, or materials screening, existing data can be used to estimate candidate properties before deciding which molecules deserve further experiments. This section compares lightweight scikit-learn models: `DummyRegressor`, `Ridge`, and `RandomForestRegressor`. The data still comes from Delaney ESOL.

## Intuition

Before modeling, build a baseline. A baseline asks: "If the model learns no structure and only predicts the training-set mean, how well does it do?" Every complex model should be compared with a baseline.

Chemically, the baseline is the reference that uses no structural information. Ridge tries to explain logS as a linear combination of descriptors. RandomForest can capture nonlinear relationships, but can also memorize local patterns in small datasets.

## Mathematical and Chemical Definitions

RMSE:

```text
RMSE = sqrt(mean((y_true - y_pred)^2))
```

MAE:

```text
MAE = mean(abs(y_true - y_pred))
```

Parity plot: true values on the x-axis and predicted values on the y-axis. Points closer to the diagonal are more accurate.

## Prepare Data and Model Tools

This part reads derived ESOL features, turns descriptors into `X`, turns logS into `y`, and creates a random train/test split. Note: random split measures interpolation within this data distribution. It does not prove extrapolation to new scaffolds.

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

sns.set_theme(style="whitegrid", context="notebook")


def mol_from_smiles(smiles):
    if pd.isna(smiles):
        return None
    return Chem.MolFromSmiles(str(smiles).strip())


def canonical_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def scaffold_smiles(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def descriptor_record(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "canonical_smiles": canonical_smiles(smiles),
        "scaffold": scaffold_smiles(mol),
    }


def build_esol_features():
    raw = pd.read_csv(RAW / "esol.csv")
    rows = []
    for row_id, row in raw.reset_index(drop=True).iterrows():
        desc = descriptor_record(row["smiles"])
        if desc is None:
            continue
        desc.update(
            {
                "row_id": row_id,
                "smiles": str(row["smiles"]).strip(),
                "logS": float(row["log solubility (mol/L)"]),
            }
        )
        rows.append(desc)
    return pd.DataFrame(rows)


def fingerprint_array(smiles, n_bits=1024):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    matrix = np.zeros((len(smiles), n_bits), dtype=np.int8)
    for idx, smi in enumerate(smiles):
        fp = generator.GetFingerprint(mol_from_smiles(smi))
        DataStructs.ConvertToNumpyArray(fp, matrix[idx])
    return matrix


DESCRIPTOR_COLUMNS = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "AromaticRings",
    "FractionCSP3",
    "HeavyAtomCount",
]

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# X is the molecular descriptor matrix; y is the experimental logS label.
esol = build_esol_features()
X = esol[DESCRIPTOR_COLUMNS].to_numpy(dtype=float)
y = esol["logS"].to_numpy(dtype=float)

# This introductory model uses random split first; the stricter scaffold split appears in the next section.
train_idx, test_idx = train_test_split(np.arange(len(esol)), test_size=0.2, random_state=RANDOM_STATE)
print(len(train_idx), "train rows;", len(test_idx), "test rows")

## Code

Compare three models: mean baseline, linear Ridge, and RandomForestRegressor.

In [ ]:
models = {
    "Baseline mean": DummyRegressor(strategy="mean"),
    "Ridge descriptors": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "RandomForest descriptors": RandomForestRegressor(n_estimators=60, random_state=RANDOM_STATE, n_jobs=-1),
}

rows = []
predictions = {}
for name, estimator in models.items():
    # clone ensures each model starts as a fresh unfitted instance, avoiding state leakage from a previous model.
    model = clone(estimator)
    model.fit(X[train_idx], y[train_idx])
    pred_train = model.predict(X[train_idx])
    pred_test = model.predict(X[test_idx])
    predictions[name] = pred_test
    rows.append(
        {
            "model": name,
            "train_RMSE": mean_squared_error(y[train_idx], pred_train) ** 0.5,
            "test_RMSE": mean_squared_error(y[test_idx], pred_test) ** 0.5,
            "test_MAE": mean_absolute_error(y[test_idx], pred_test),
            "test_R2": r2_score(y[test_idx], pred_test),
        }
    )

results = pd.DataFrame(rows).sort_values("test_RMSE")
results.round(3)

## Read the Parity Plot

This cell selects the model with the lowest test RMSE and plots true logS against predicted logS. The diagonal is not a fitted line; it is the perfect-prediction line. Larger distance from the diagonal means larger error.

In [ ]:
best_name = results.iloc[0]["model"]
plot_df = pd.DataFrame(
    {
        "true_logS": y[test_idx],
        "pred_logS": predictions[best_name],
        "smiles": esol.iloc[test_idx]["smiles"].to_numpy(),
    }
)

plt.figure(figsize=(5, 5))
sns.scatterplot(data=plot_df, x="true_logS", y="pred_logS", alpha=0.7)
lims = [min(plot_df["true_logS"].min(), plot_df["pred_logS"].min()), max(plot_df["true_logS"].max(), plot_df["pred_logS"].max())]
plt.plot(lims, lims, "k--", linewidth=1)
plt.title(f"Parity plot: {best_name}")
plt.tight_layout()

## Find the Most Difficult Molecules

Average model scores only describe overall error. Also inspect failure cases: are they larger, more hydrophobic, unusual in functional groups, or located in rare regions of the training data?

In [ ]:
# abs_error helps find the molecules with the largest prediction deviations in this test split.
plot_df["abs_error"] = (plot_df["true_logS"] - plot_df["pred_logS"]).abs()
worst = plot_df.sort_values("abs_error", ascending=False).head(6)
display(worst[["true_logS", "pred_logS", "abs_error", "smiles"]].round(3))

Draw.MolsToGridImage(
    [mol_from_smiles(s) for s in worst["smiles"]],
    legends=[f"err={e:.2f}" for e in worst["abs_error"]],
    molsPerRow=3,
    subImgSize=(260, 180),
)

## Observation Questions

1. Is RandomForestRegressor always better than Ridge?
2. What might it mean if train RMSE and test RMSE differ a lot?
3. What common structures or descriptor features appear in the worst predictions?

### Hints

1. Not always. RandomForest can model nonlinearity, but it is not guaranteed to win on small data, weak features, or out-of-distribution tests.
2. Very low train error and much higher test error often suggest overfitting, or a distribution difference between train and test.
3. Check MolWt, MolLogP, TPSA, HBD/HBA, and whether the molecule has large rings, many heteroatoms, strongly hydrophobic fragments, or structures rare in the training set.

## Summary

Property prediction should start from a baseline and then compare models. RMSE and parity plots are entry points for reading regression results, but they do not answer whether a model extrapolates to new scaffolds.